# Noisy-LJ rollout in a normalized latent space

This experiment isolates noisy LJ from the shared simulator. Both the two-dimensional autoencoder and the latent propagator are trained only on noisy-LJ trajectories. Positions and predicted displacements use the fixed normalized coordinate system, while the encoder and propagator retain physical reference-graph context reconstructed from the stored length scale. The autonomous propagator observes only $z(1)$ and $z(5)$, receives no frame/progress variable, and is trained with one-step supervision.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT/'src'/'lss').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT/'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT/'src'))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lss.latent.experiment import run_latent_experiment, seed_everything
from lss.latent.simulation import pearson_r, r2_score
from lss.plotting import PAPER_COLORS, apply_editorial_style, dataset_color, style_axes
from lss.utils import resolve_device
apply_editorial_style()
plt.rcParams.update({'figure.dpi': 180, 'savefig.dpi': 400})
DEVICE = resolve_device('auto')
print(f'Device: {DEVICE}')

## Configuration

In [ ]:
SEED = 34234
FORCE_TRAIN = False
TRAIN_NETWORKS = 100
VAL_NETWORKS = 50
TRAIN_STEPS_PER_SIM = 100
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
LATENT_DIM = 2
OBSERVED_FRAMES = (1, 5)
ROLLOUT_STEPS = [25, 50, 75, 100, 125, 150, 175, 199]
assert max(OBSERVED_FRAMES) <= 5

DATA = PROJECT_ROOT/'data'/'lj-noisy-eps0.01-sigma1.0-cutoff1.122_200sims_200frames.pt'
if not DATA.is_file():
    raise FileNotFoundError(DATA)
OUTPUT = PROJECT_ROOT/'notebooks'/'results'/'07_noisy_lj_normalized_context_rollout'
OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE = OUTPUT/'noisy_lj_2d_matched_budget_z1_z5_onestep.pt'

AE_CONFIG = {
    'latent_dim': LATENT_DIM, 'latent_tokens': 32, 'hidden_size': 64,
    'model': 'single_stage_attention', 'edge_feature_dim': 13, 'batch_graphs': 256,
    'target_mode': 'normalized_delta', 'node_feature_mode': 'normalized_delta',
    'max_train_frames_per_sim': TRAIN_STEPS_PER_SIM + 1, 'val_frame_skip': 1,
    'max_val_frames_per_sim': TRAIN_STEPS_PER_SIM + 1,
    'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'balance_sources': False, 'mix_sources': False,
}
PROPAGATOR_CONFIG = {
    'max_train_transitions_per_sim': TRAIN_STEPS_PER_SIM,
    'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'hidden_size': 64, 'objective': 'fixed_history_one_step', 'model': 'fixed_velocity_residual_mlp',
    'loss': 'delta', 'step_stride': 1, 'fixed_observed_frames': OBSERVED_FRAMES, 'multistep_horizons': [1],
    'train_trajectories_per_source': {'lj_noisy': TRAIN_NETWORKS},
    'val_trajectories_per_source': {'lj_noisy': VAL_NETWORKS},
    'mix_sources': False, 'balance_sources': False, 'use_static_context': True,
    'context_pool': 'moments', 'context_dim': 32, 'rollout_eval_every_epoch': True,
    'rollout_eval_source': 'lj_noisy', 'rollout_eval_horizons': [100, 150],
    'rollout_eval_sims_per_source': 20,
}
print({
    'data': str(DATA), 'train': TRAIN_NETWORKS, 'validation': VAL_NETWORKS,
    'test': 200-TRAIN_NETWORKS-VAL_NETWORKS, 'latent_dim': LATENT_DIM,
    'observed_frames': OBSERVED_FRAMES, 'physical_static_context': True,
    'cache': str(MODEL_CACHE),
})

## Train the noisy-LJ autoencoder and propagator

In [ ]:
experiment_config = {
    'dataset_name': 'lj_noisy', 'train_count': TRAIN_NETWORKS, 'val_count': VAL_NETWORKS,
    'split_seed': SEED, 'model_seed': SEED, 'repeat_idx': 1,
    'split_stratify_temperature': False, 'min_train_p_ratio': None,
    'device': str(DEVICE), 'pos_dim': 2, 'frame_skip': 1, 'train_frame_start_order': 0,
    'edge_multiplicity': 1, 'edge_vector_dim': 2, 'edge_mode': 'stored',
    'coordinate_normalization': 'position_normalization',
    'reference_context_mode': 'physical',
    'edge_feature_schema': 'physical_static_normalized_edge_changes_v3',
    'ae_config': AE_CONFIG, 'propagator_config': PROPAGATOR_CONFIG,
    'rollout_steps_grid': ROLLOUT_STEPS,
    'rollout_eval_splits': ['test'], 'rollout_eval_max_sims_by_split': {'test': 50},
    'rollout_eval_source': 'lj_noisy',
    'early_stop_min_delta': 1e-5,
    'should_rollout': True, 'should_train_propagator': True,
    'force_train': FORCE_TRAIN, 'force_train_autoencoder': FORCE_TRAIN,
    'cache_path': str(MODEL_CACHE), 'cache_require_matching_config': True,
}
source = {
    'dataset_name': 'lj_noisy', 'source_name': 'lj_noisy',
    'label': 'noisy LJ | normalized coordinates + physical context',
    'path': str(DATA), 'train_count': TRAIN_NETWORKS, 'val_count': VAL_NETWORKS,
    'edge_multiplicity': 1, 'edge_vector_dim': 2,
}
seed_everything(SEED)
result = run_latent_experiment(source, experiment_config, device=DEVICE)
display(result['split_info'])
display(result['ae_history'].tail())
display(result['dyn_history'].tail())

## Normalization and context audit

In [ ]:
graph = result['train_data'][0][0]
bounds = np.array([graph.box.x1, graph.box.x2, graph.box.y1, graph.box.y2], dtype=float)
scale = float(torch.as_tensor(graph.reference_length_scale).reshape(-1)[0])
audit = pd.DataFrame([{
    'coordinate_system': getattr(graph, 'coordinate_normalization', 'missing'),
    'reference_context_mode': getattr(graph, 'reference_context_mode', 'missing'),
    'x_min': bounds[0], 'x_max': bounds[1], 'y_min': bounds[2], 'y_max': bounds[3],
    'stored_physical_length_scale': scale,
    'max_observed_frame': max(result['params']['fixed_observed_frames']),
    'latent_dim': result['params']['latent_dim'],
}])
assert audit.max_observed_frame.iloc[0] <= 5
assert audit.reference_context_mode.iloc[0] == 'physical'
assert np.isfinite(scale) and scale > 0
display(audit.round(4))

## Held-out rollout performance

In [ ]:
rollout = result['rollout_rows'].query("split == 'test'").copy()
direct = result['ae_reconstruction_rows'].query("split == 'test'").copy()
summary_rows = []
for representation, frame in [('direct Autoencoder', direct), ('latent rollout', rollout)]:
    for estimator, true_col, pred_col in [
        ('directional-side endpoint', 'endpoint_true_p_ratio', 'endpoint_pred_p_ratio'),
        ('trajectory fit', 'true_p_ratio', 'pred_p_ratio'),
    ]:
        for step, group in frame.groupby('rollout_steps'):
            valid = group[[true_col, pred_col]].replace([np.inf, -np.inf], np.nan).dropna()
            errors = group.final_pos_mse.replace([np.inf, -np.inf], np.nan).dropna()
            summary_rows.append({
                'representation': representation, 'estimator': estimator,
                'rollout_steps': int(step), 'n_networks': len(valid),
                'p_ratio_r2': r2_score(valid[true_col], valid[pred_col]) if len(valid) > 1 else np.nan,
                'pearson_r': pearson_r(valid[true_col], valid[pred_col]) if len(valid) > 1 else np.nan,
                'position_mse': errors.mean(),
            })
summary = pd.DataFrame(summary_rows)
display(summary[summary.rollout_steps.isin([50, 100, 150, 199])].round(4))

headline = summary[summary.estimator.eq('directional-side endpoint')]
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), constrained_layout=True)
for representation, color, marker, linestyle in [
    ('direct Autoencoder', PAPER_COLORS['blue'], 'o', '--'),
    ('latent rollout', dataset_color('lj_noisy'), 's', '-'),
]:
    group = headline[headline.representation.eq(representation)].sort_values('rollout_steps')
    axes[0].plot(group.rollout_steps, group.p_ratio_r2, color=color, marker=marker,
                 linestyle=linestyle, linewidth=1.9, markersize=4.5, label=representation)
rollout_error = headline[headline.representation.eq('latent rollout')].sort_values('rollout_steps')
axes[1].plot(rollout_error.rollout_steps, rollout_error.position_mse,
             color=dataset_color('lj_noisy'), marker='s', linewidth=1.9, markersize=4.5)
axes[0].axhline(0, color=PAPER_COLORS['slate'], linewidth=.8)
style_axes(axes[0], xlabel='rollout step', ylabel='p-ratio R²', legend=True)
style_axes(axes[1], xlabel='rollout step', ylabel='position MSE', legend=False)
plt.show()

## Held-out network scatter

In [ ]:
SCATTER_STEPS = [50, 100, 199]
fig, axes = plt.subplots(1, len(SCATTER_STEPS), figsize=(10.8, 3.6), constrained_layout=True)
for ax, step in zip(axes, SCATTER_STEPS):
    group = rollout[rollout.rollout_steps.eq(step)].replace([np.inf, -np.inf], np.nan).dropna(
        subset=['endpoint_true_p_ratio', 'endpoint_pred_p_ratio'])
    true = group.endpoint_true_p_ratio.to_numpy(float)
    pred = group.endpoint_pred_p_ratio.to_numpy(float)
    ax.scatter(true, pred, s=25, alpha=.7, color=dataset_color('lj_noisy'), edgecolor='none')
    if len(group):
        lo, hi = np.nanmin(np.r_[true, pred]), np.nanmax(np.r_[true, pred])
        pad = .04 * max(hi-lo, 1e-6)
        ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], '--', color=PAPER_COLORS['ink'], linewidth=1)
        score = r2_score(true, pred)
        ax.text(.04, .96, f'R² = {score:.3f}\nN = {len(group)}', transform=ax.transAxes, va='top')
    style_axes(ax, xlabel=f'true p-ratio\nframe {step}',
               ylabel='predicted p-ratio' if ax is axes[0] else '', legend=False)
plt.show()